In [ ]:
!pip install streamlit pyngrok scikit-learn pandas numpy plotly

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
import pickle

# Create dummy dataset
np.random.seed(42)
data = pd.DataFrame({
    "age": np.random.randint(20, 80, 500),
    "chol": np.random.randint(150, 300, 500),
    "bp": np.random.randint(80, 180, 500),
})

data["target"] = (
    (data["age"] > 50) &
    (data["chol"] > 220) &
    (data["bp"] > 130)
).astype(int)

X = data[["age", "chol", "bp"]]
y = data["target"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

model = RandomForestClassifier()
model.fit(X_train, y_train)

print("Model Accuracy:", model.score(X_test, y_test))

# Save model
with open("heart.pkl", "wb") as f:
    pickle.dump(model, f)

print("Model Saved Successfully ✅")

Model Accuracy: 1.0
Model Saved Successfully ✅


In [ ]:
%%writefile app.py

import streamlit as st
import pandas as pd
import plotly.express as px

st.set_page_config(page_title="PreMedix", layout="wide")

st.title("PreMedix - AI Healthcare Assistant")

# Sidebar Menu
menu = st.sidebar.selectbox(
    "Select Feature",
    ["Heart Disease", "Diabetes", "Stroke", "Symptom Checker", "Health Dashboard"]
)

# ================= HEART DISEASE =================
if menu == "Heart Disease":
    st.header(" Heart Disease Prediction")

    age = st.number_input("Age", 18, 100, 40)
    chol = st.number_input("Cholesterol", 100, 400, 200)
    bp = st.number_input("Blood Pressure", 80, 200, 120)

    if st.button("Predict Heart Risk"):

        risk_score = (age/100) + (chol/400) + (bp/200)
        percentage = min(int(risk_score * 40), 100)

        st.subheader("Risk Percentage")
        st.progress(percentage)
        st.write(f"Predicted Risk: {percentage}%")

        if percentage > 60:
            st.error("⚠ High Risk of Heart Disease")
        elif percentage > 40:
            st.warning("⚠ Moderate Risk")
        else:
            st.success("Low Risk")

        st.markdown("###  Why this prediction?")
        st.write("""
        • Higher age increases cardiovascular risk
        • Elevated cholesterol contributes to artery blockage
        • High blood pressure increases stroke probability
        """)

        if percentage > 60:
            st.markdown("###  Recommended Actions")
            st.write("""
            - Reduce salt intake
            - Exercise 30 mins daily
            - Monitor blood pressure weekly
            - Avoid smoking and alcohol
            """)

# ================= DIABETES =================
elif menu == "Diabetes":
    st.header(" Diabetes Prediction")

    glucose = st.number_input("Glucose Level", 70, 200, 110)
    bmi = st.number_input("BMI", 15.0, 40.0, 22.0)

    if st.button("Predict Diabetes Risk"):
        risk = (glucose/200) + (bmi/40)
        percentage = min(int(risk * 50), 100)

        st.progress(percentage)
        st.write(f"Predicted Risk: {percentage}%")

        if percentage > 60:
            st.error("⚠ High Risk of Diabetes")
        elif percentage > 40:
            st.warning("⚠ Moderate Risk")
        else:
            st.success("Low Risk")

# ================= STROKE =================
elif menu == "Stroke":
    st.header(" Stroke Prediction")

    age = st.number_input("Age", 18, 100, 40)
    hypertension = st.radio("Hypertension?", ["No", "Yes"])

    if st.button("Predict Stroke Risk"):
        percentage = 0

        if age > 60:
            percentage += 40
        if hypertension == "Yes":
            percentage += 50

        st.progress(percentage)
        st.write(f"Predicted Risk: {percentage}%")

        if percentage > 60:
            st.error("⚠ High Risk of Stroke")
        else:
            st.success(" Low Risk")

# ================= SYMPTOM CHECKER =================
elif menu == "Symptom Checker":
    st.header(" Symptom Checker")

    fever = st.checkbox("Fever")
    cough = st.checkbox("Cough")
    fatigue = st.checkbox("Fatigue")
    chest_pain = st.checkbox("Chest Pain")

    if st.button("Check Symptoms"):

        if fever and cough:
            st.warning("Possible Infection or Flu")
        elif chest_pain:
            st.error("Seek Medical Attention Immediately")
        elif fatigue:
            st.info("Possible Stress or Vitamin Deficiency")
        else:
            st.success("No major risk detected")

# ================= DASHBOARD =================
elif menu == "Health Dashboard":
    st.header(" Health Analytics Dashboard")

    df = pd.DataFrame({
        "Disease": ["Heart", "Diabetes", "Stroke"],
        "Risk (%)": [65, 45, 30]
    })

    fig = px.bar(df, x="Disease", y="Risk (%)", color="Disease")
    st.plotly_chart(fig)

Overwriting app.py


In [ ]:
!pip install pyngrok

In [ ]:
!ngrok config add-authtoken 3AHyztWWDinvHECWXjTkuqSMbEP_33EuCFHtBqnRWq13X1Ein

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [ ]:
from pyngrok import ngrok
import subprocess
import time

# Terminate any existing ngrok tunnels
ngrok.kill()

# Start streamlit
process = subprocess.Popen(["streamlit", "run", "app.py"])

# Wait for streamlit to start
time.sleep(5)

# Connect ngrok
public_url = ngrok.connect(8501)
print("Your App URL:", public_url)

Your App URL: NgrokTunnel: "https://nonbrutal-unpublicly-maye.ngrok-free.dev" -> "http://localhost:8501"
